# OpenShift AI Lab — Redaction & Discovery walkthrough

Run this notebook in an **OpenShift AI Workbench** to see how the agents call MinIO and catalog models.

Set env vars `REDACT_API_URL` and `DISCOVERY_API_URL`, or use the in-cluster defaults in the next cell.

In [ ]:
import os, json, subprocess, sys

try:
    import httpx
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "httpx"])
    import httpx

REDACT = os.getenv("REDACT_API_URL", "http://redaction-agent.redaction-agent.svc.cluster.local:8000")
DISCOVERY = os.getenv("DISCOVERY_API_URL", "http://discovery-agent.discovery-agent.svc.cluster.local:8001")
client = httpx.Client(timeout=120.0, verify=False)
print("Redaction:", REDACT)
print("Discovery:", DISCOVERY)
print("redact health:", client.get(f"{REDACT}/healthz").json())
print("discovery health:", client.get(f"{DISCOVERY}/healthz").json())


In [ ]:
docs = client.get(f"{REDACT}/documents").json()
print(len(docs), "documents")
for d in docs[:10]:
    print("-", d.get("key"), d.get("size"))

In [ ]:
pdfs = [d["key"] for d in docs if str(d.get("key","")).lower().endswith(".pdf")]
if pdfs:
    job = client.post(f"{REDACT}/redact", json={
        "documents": [pdfs[0]],
        "person": "Jordan Hale",
        "place": "Plant B",
        "time": "July 2021",
        "events": "The chemical spill at Plant B in July 2021",
        "custom": "",
    }).json()
    print(job.get("status"), job.get("progress"))
    print(json.dumps(job.get("results"), indent=2)[:2000])
else:
    print("No PDFs — seed MinIO first")

In [ ]:
print(client.post(f"{DISCOVERY}/index", json={}).json())
found = client.post(f"{DISCOVERY}/search", json={
    "query": "chemical spill at Plant B",
    "top_k": 3,
    "summarize": True,
}).json()
print(json.dumps(found, indent=2)[:3000])

## Next
- Streamlit UI (Redaction + Discovery tabs)
- Observability / Tempo traces for `redaction-agent` and `discovery-agent`
- `scripts/run_load_test.sh` from your laptop